### Instalação das dependências

Os pacotes abaixo são necessários para a manipulação de séries temporais com Pandas. Execute esta célula antes de qualquer outra.

In [ ]:
!pip install pandas -q

# Código 1.5 — Agregação Temporal Segura com Pandas

**Objetivo:** Criar features agregadas baseadas em tempo (como médias móveis) sem provocar vazamento do futuro para o passado (*look-ahead bias*).

### Passo 1: Criação da Série Temporal Sintética

Simulamos leituras diárias de sensores em equipamentos.

In [ ]:
import pandas as pd
import numpy as np

# Criar datas diárias de 30 dias para 2 equipamentos
np.random.seed(42)
datas = pd.date_range(start="2024-01-01", periods=15, freq="D")
data_eq1 = pd.DataFrame({
    "data": datas,
    "equipamento_id": "EQ-01",
    "temperatura": np.random.normal(loc=70, scale=5, size=15)
})
data_eq2 = pd.DataFrame({
    "data": datas,
    "equipamento_id": "EQ-02",
    "temperatura": np.random.normal(loc=85, scale=8, size=15)
})

df_sensores = pd.concat([data_eq1, data_eq2]).sort_values(["equipamento_id", "data"]).reset_index(drop=True)
print("Amostra dos dados brutos:")
print(df_sensores.head(6))

**Entendendo os parâmetros de geração da série temporal:**
- `pd.date_range(start="2024-01-01", periods=15, freq="D")`: Gera uma sequência contínua de 15 datas diárias a partir do dia 01/01/2024.
- `np.random.normal(loc=70, scale=5, size=15)`: Amostra valores da distribuição normal com média 70 (`loc`) e desvio padrão 5 (`scale`).

### Passo 2: Agregação por Janela Móvel Deslocada (`shift` + `rolling`)

Utilizamos `.shift(1)` para garantir que o cálculo da média móvel dos últimos 3 dias utilize estritamente observações passadas.

In [ ]:
# Agrupar por equipamento e ordenar por data
df_sensores = df_sensores.sort_values(["equipamento_id", "data"])

# Agregação segura: shift(1) garante que o dia atual NÃO entra na média móvel do passado
df_sensores["temp_media_3d_passado"] = (
    df_sensores.groupby("equipamento_id")["temperatura"]
    .transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())
)

print("Exemplo de Agregação Segura (Sem Look-Ahead Bias):")
print(df_sensores[["equipamento_id", "data", "temperatura", "temp_media_3d_passado"]].head(8))

**Entendendo as funções de janela móvel segura:**
- `groupby("equipamento_id")`: Garante que as janelas de tempo não misturem dados de equipamentos diferentes.
- `.shift(1)`: Desloca a série temporal uma posição para baixo, excluindo o próprio dia atual do cálculo.
- `.rolling(window=3, min_periods=1)`: Define a janela móvel de 3 dias passados, aceitando calcular a média se houver pelo menos 1 dia histórico disponível.

---

## Conclusão da Aula — Agregação Temporal Segura

O uso de `.shift(1)` em agregações temporais (como médias móveis) é fundamental para evitar a contaminação de informações do futuro no momento do treino (*look-ahead bias*), garantindo estimativas realistas da performance do modelo.